# 08 — Meta-Learning v1: Four-Agent Supervisory Ensemble

Orchestrates four heterogeneous base agents (LGBM, TCN, DRL v3, GP v2) into a
unified López de Prado–style meta-labeling framework.

| Layer | Agent | Signal type | Key improvement |
|---|---|---|---|
| Base | LGBM v12 | P(up) ∈ [0,1] | Best performer — primary execution model |
| Base | TCN v0 | P(up), P(down) ∈ [0,1] | Sequential feature model |
| Base | DRL PPO v3 | Action ∈ {-1, 0, 1} | min_hold=6 hard constraint (vs v2 penalty overcorrection) |
| Base | GP v2 | Action ∈ {-1, 0, 1} | Boolean-gate-only pset (vs v1 coefficient inflation) |
| **Supervisor** | **MetaAgent v1** | **Position ∈ [-1, 1]** | **4 agents — adds GP v2 as 4th decorrelated signal** |

**v1 changes vs v0:**
- DRL: updated to v3 (min_hold constraint replaces overcorrected penalty) — artifact `07_drl_omni_0fee_v3`
- GP: added as 4th base agent — artifact `09_gp_omni_0fee_v2`
- Meta features: `gp_action` added to supervisor's feature set
- `build_signal_df()` now includes GP signals in primary signal generation

**Pipeline:**
1. Load or recompute base-agent OOS signals
2. Merge into a unified `signals_df` with GP as 4th column
3. Walk-forward meta-labeling (Triple Barrier on signal bars, LightGBM meta-classifier)
4. Continuous position sizing from meta-probability
5. Benchmark vs each standalone agent


In [1]:
import calendar
import json
import time
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from hmats.agents.lgbm_agent import LGBMAgent
from hmats.agents.tcn_agent  import TCNAgent, TCNConfig
from hmats.agents.drl_agent  import DRLAgent, _DEFAULT_DRL_FEATURES
from hmats.agents.gp_agent   import GPTradingAgent, GP_FEATURES
from hmats.agents.meta_agent import MetaSupervisoryAgent, build_signal_df, run_sized_backtest
from hmats.viz.plots import plot_equity_drawdown, save_fig

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

try:    plt.style.use('seaborn-v0_8-whitegrid')
except: plt.style.use('seaborn-whitegrid')
mpl.rcParams.update({
    'font.family': 'serif', 'font.serif': ['DejaVu Serif'],
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelsize': 10, 'axes.titlesize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'legend.framealpha': 0.85,
    'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})
ACCENT='#F7931A'; BLUE='#2962FF'; GREY='#9E9E9E'
RED='#EF5350'; GREEN='#26A69A'; PURPLE='#7B1FA2'; TEAL='#00ACC1'

# ── Time splits ───────────────────────────────────────────────────────────────
# Base agents generate signals from META_SIGNAL_START (WFO history for meta-training)
# Meta-model is evaluated on OOS_START onward
META_SIGNAL_START = pd.Timestamp('2022-01-01')  # start of base-agent signal history
OOS_START         = pd.Timestamp('2024-01-01')  # meta-model OOS / final evaluation

LGBM_LONG_THR  = 0.58
LGBM_SHORT_THR = 0.35

print(f'Meta signal history : {META_SIGNAL_START.date()} → ...')
print(f'Meta OOS start      : {OOS_START.date()}')
print('Base agents         : LGBM v12 | TCN v0 | DRL v3 | GP v2')
print('Imports OK')

Meta signal history : 2022-01-01 → ...
Meta OOS start      : 2024-01-01
Base agents         : LGBM v12 | TCN v0 | DRL v3 | GP v2
Imports OK


In [2]:
def _find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise RuntimeError('repo root not found')

REPO_DIR  = _find_repo_root()
FEAT_DIR  = REPO_DIR / 'data' / 'features'
EXT_DIR   = REPO_DIR / 'data' / 'external'
ARTS_DIR  = REPO_DIR / 'artifacts' / '08_meta_learning_v1'
CACHE_DIR = ARTS_DIR / 'signal_cache'
ARTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Pre-computed signals from individual notebooks
LGBM_ARTS = REPO_DIR / 'artifacts' / '02_lgbm_omni_0fee_v12'
DRL_ARTS  = REPO_DIR / 'artifacts' / '07_drl_omni_0fee_v3'
GP_ARTS   = REPO_DIR / 'artifacts' / '09_gp_omni_0fee_v2'
print(f'Artifacts → {ARTS_DIR}')

Artifacts → /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/artifacts/08_meta_learning_v1


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
print('Loading V1 features...')
v1 = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_features.parquet')
v1.index = v1.index.tz_localize(None) if v1.index.tz else v1.index

print('Loading V4 features...')
v4 = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet')
v4.index = v4.index.tz_localize(None) if v4.index.tz else v4.index

merged = v1.copy()

_raw = pd.read_parquet(REPO_DIR / 'data' / 'raw' / 'BTCUSDT_1h.parquet')
_raw.index = _raw.index.tz_convert(None)
merged['high'] = _raw['high'].reindex(merged.index)
merged['low']  = _raw['low'].reindex(merged.index)

for col in ['close_vs_true_vwap', 'hurst_24h', 'hurst_72h', 'tfi_pct', 'tfi_z_24h',
            'bb_width_pct', 'sideways_flag']:
    if col in v4.columns:
        merged[col] = v4[col].reindex(merged.index)

oos_mask = merged.index >= OOS_START
oos_df   = merged[oos_mask].copy()

try:
    struct = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_structural.parquet')
    struct.index = struct.index.tz_convert(None) if struct.index.tz else struct.index
    for col in ['liq_vwap_dev_24h', 'volat_atr_20_pct', 'mtf_alignment', 'mtf_h4_rsi']:
        if col in struct.columns:
            merged[col] = struct[col].reindex(merged.index)
    print('Structural features loaded')
except Exception as e:
    print(f'Structural features skipped: {e}')

print(f'Total bars: {len(merged):,}  |  OOS bars: {len(oos_df):,}')

Loading V1 features...
Loading V4 features...
Structural features loaded
Total bars: 74,366  |  OOS bars: 20,785


: 

In [ ]:
# ── PHASE 1: Generate / load base-agent signals ───────────────────────────────
# Signals are cached to disk to avoid re-running expensive WFO loops.
# Delete the cache files to force regeneration.
print('='*60)
print('PHASE 1 — Base-agent signal generation')
print('='*60)

# ── 1A. LGBM signals ──────────────────────────────────────────────────────────
lgbm_cache = CACHE_DIR / 'lgbm_signals.parquet'
if lgbm_cache.exists():
    print(f'[LGBM] Loading from cache: {lgbm_cache}')
    lgbm_p_up = pd.read_parquet(lgbm_cache)['lgbm_p_up']
else:
    print('[LGBM] Running M1Y WFO (full history for meta-training)...')
    t0 = time.time()
    lgbm_agent = LGBMAgent()
    lgbm_p_up = lgbm_agent.generate_signals(
        merged, oos_start=META_SIGNAL_START, verbose=True, full_history=True
    )
    lgbm_p_up = lgbm_p_up[lgbm_p_up.index >= META_SIGNAL_START]
    lgbm_p_up.to_frame().to_parquet(lgbm_cache)
    print(f'[LGBM] Done in {(time.time()-t0)/60:.1f} min  →  cached')

print(f'[LGBM] Signals: {len(lgbm_p_up):,} bars  '
      f'({lgbm_p_up.index[0].date()} → {lgbm_p_up.index[-1].date()})')
print(f'[LGBM] Mean P(up)={lgbm_p_up.mean():.3f}  '
      f'Long signals: {(lgbm_p_up > LGBM_LONG_THR).sum():,}  '
      f'Short signals: {(lgbm_p_up < LGBM_SHORT_THR).sum():,}')

In [ ]:
# ── 1B. TCN signals ───────────────────────────────────────────────────────────
tcn_cache = CACHE_DIR / 'tcn_signals.parquet'
if tcn_cache.exists():
    print(f'[TCN] Loading from cache: {tcn_cache}')
    tcn_signals = pd.read_parquet(tcn_cache)
else:
    print('[TCN] Training on pre-OOS data, then inferring from META_SIGNAL_START...')
    t0 = time.time()
    tcn_cfg = TCNConfig(epochs=80, patience=15)
    tcn_agent = TCNAgent(cfg=tcn_cfg)
    train_result = tcn_agent.train(
        merged,
        train_end=META_SIGNAL_START,   # train on data before 2022
        verbose=True,
        save_path=CACHE_DIR / 'tcn_v0_model.pt',
    )
    print(f'[TCN] Training done in {(time.time()-t0)/60:.1f} min  '
          f'best_val_loss={train_result["best_val_loss"]:.5f}')

    t1 = time.time()
    tcn_signals = tcn_agent.generate_signals(
        merged, oos_start=META_SIGNAL_START, verbose=True
    )
    tcn_signals.to_parquet(tcn_cache)
    print(f'[TCN] Inference done in {(time.time()-t1)/60:.1f} min  →  cached')

print(f'[TCN] Signals: {len(tcn_signals):,} bars  '
      f'({tcn_signals.index[0].date()} → {tcn_signals.index[-1].date()})')

In [ ]:
# ── 1C. DRL signals ───────────────────────────────────────────────────────────
drl_cache = CACHE_DIR / 'drl_signals.parquet'

# Check if 07_ notebook already produced OOS signals
drl_precomputed = DRL_ARTS / 'drl_oos_signals.parquet'

if drl_cache.exists():
    print(f'[DRL] Loading from cache: {drl_cache}')
    drl_action = pd.read_parquet(drl_cache)['drl_action']
elif drl_precomputed.exists():
    print(f'[DRL] Loading pre-computed OOS signals from 07_ notebook...')
    drl_action = pd.read_parquet(drl_precomputed)['drl_action']
    drl_action.to_frame().to_parquet(drl_cache)
else:
    print('[DRL] Running M1Y WFO (full history for meta-training)...')
    t0 = time.time()
    drl_feats = [f for f in _DEFAULT_DRL_FEATURES if f in merged.columns]
    drl_agent = DRLAgent(
        features=drl_feats,
        window_size=24,
        train_window_h=8760,
        step_size=720,
        total_timesteps=300_000,
    )
    drl_action = drl_agent.generate_signals(
        merged, oos_start=META_SIGNAL_START, verbose=True, full_history=True
    )
    drl_action = drl_action[drl_action.index >= META_SIGNAL_START]
    drl_action.to_frame().to_parquet(drl_cache)
    print(f'[DRL] Done in {(time.time()-t0)/60:.1f} min  →  cached')

print(f'[DRL] Signals: {len(drl_action):,} bars  '
      f'({drl_action.index[0].date()} → {drl_action.index[-1].date()})')
vc = drl_action.value_counts().sort_index()
for k, v in vc.items():
    print(f'  {str(k):>3}: {v:6,}  ({v/len(drl_action)*100:.1f}%)')

In [ ]:
# ── 1D. GP signals ───────────────────────────────────────────────────────────
gp_cache = CACHE_DIR / 'gp_signals.parquet'
gp_precomputed = GP_ARTS / 'gp_oos_signals.parquet'

if gp_cache.exists():
    print(f'[GP] Loading from cache: {gp_cache}')
    gp_action = pd.read_parquet(gp_cache)['gp_action']
elif gp_precomputed.exists():
    print(f'[GP] Loading pre-computed OOS signals from 09_gp_omni_0fee_v2 notebook...')
    gp_action = pd.read_parquet(gp_precomputed)['gp_action']
    gp_action.to_frame().to_parquet(gp_cache)
else:
    print('[GP] Running M1Y WFO (full history for meta-training)...')
    t0 = time.time()
    gp_feats = [f for f in GP_FEATURES if f in merged.columns]
    gp_agent = GPTradingAgent(
        features=gp_feats,
        population_size=400, generations=40,
        parsimony_coefficient=0.002,
        train_window_h=8760, step_size=336,  # 2-week cadence (matches 09_gp_omni_0fee_v2)
        cx_prob=0.40, mut_prob=0.40,
        flat_threshold=0.5, logic_only=True,
    )
    gp_action = gp_agent.generate_signals(
        merged, oos_start=META_SIGNAL_START, verbose=True
    )
    gp_action = gp_action[gp_action.index >= META_SIGNAL_START]
    gp_action.to_frame().to_parquet(gp_cache)
    print(f'[GP] Done in {(time.time()-t0)/60:.1f} min  →  cached')

print(f'[GP] Signals: {len(gp_action):,} bars  '
      f'({gp_action.index[0].date()} → {gp_action.index[-1].date()})')
vc = gp_action.value_counts().sort_index()
for k, v in vc.items():
    name = {-1: 'Short', 0: 'Flat', 1: 'Long'}.get(k, str(k))
    print(f'  {name:5}: {v:6,}  ({v/len(gp_action)*100:.1f}%)')


In [ ]:
# ── PHASE 2: Build unified signal DataFrame ───────────────────────────────────
print('='*60)
print('PHASE 2 — Signal alignment and combination')
print('='*60)

# Common index: intersection of all signal periods and the full data range
common_start = max(
    lgbm_p_up.dropna().index[0],
    tcn_signals.dropna().index[0] if len(tcn_signals.dropna()) > 0 else META_SIGNAL_START,
    gp_action.dropna().index[0] if len(gp_action.dropna()) > 0 else META_SIGNAL_START,
    META_SIGNAL_START,
)
print(f'Common signal start: {common_start.date()}')

ohlcv_common = merged[merged.index >= common_start].copy()

signals_df = build_signal_df(
    ohlcv_df=ohlcv_common,
    lgbm_p_up=lgbm_p_up,
    tcn_signals=tcn_signals,
    drl_action=drl_action,
    gp_action=gp_action,
    long_threshold=LGBM_LONG_THR,
    short_threshold=LGBM_SHORT_THR,
)

print(f'signals_df: {signals_df.shape}  '
      f'({signals_df.index[0].date()} → {signals_df.index[-1].date()})')

n_primary = signals_df['primary_signal'].sum()
n_long    = (signals_df['primary_side'] ==  1).sum()
n_short   = (signals_df['primary_side'] == -1).sum()
print(f'Primary signals: {n_primary:,} total  (Long={n_long:,}  Short={n_short:,})')
print(f'Signal rate: {n_primary/len(signals_df)*100:.1f}%')

# Preview signal agreement
print('\nSignal co-occurrence (primary = any agent fires):')
sig_meta = signals_df[signals_df.index >= OOS_START]
for direction, side in [('Long', 1), ('Short', -1)]:
    mask = sig_meta['primary_side'] == side
    sub = sig_meta[mask]
    if len(sub) == 0: continue
    lgbm_agree = ((sub['lgbm_p_up'] > LGBM_LONG_THR) if side==1 else (sub['lgbm_p_up'] < LGBM_SHORT_THR)).sum()
    tcn_agree  = ((sub['tcn_p_up'] > LGBM_LONG_THR) if side==1 else (sub['tcn_p_down'] > (1-LGBM_SHORT_THR))).sum()
    drl_agree  = (sub['drl_action'] == side).sum()
    gp_agree   = (sub['gp_action']  == side).sum()
    print(f'  {direction:6}: {len(sub):5} signals  '
          f'LGBM={lgbm_agree:4} ({lgbm_agree/max(len(sub),1)*100:.0f}%)  '
          f'TCN={tcn_agree:4} ({tcn_agree/max(len(sub),1)*100:.0f}%)  '
          f'DRL={drl_agree:4} ({drl_agree/max(len(sub),1)*100:.0f}%)  '
          f'GP={gp_agree:4} ({gp_agree/max(len(sub),1)*100:.0f}%)')

In [ ]:
# ── Signal correlation heatmap ────────────────────────────────────────────────
sig_cols = ['lgbm_p_up', 'tcn_p_up', 'tcn_p_down', 'drl_action', 'gp_action']
corr_df  = signals_df[sig_cols].dropna()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_df.corr(method='spearman'), ax=ax,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.3f', linewidths=0.5,
            cbar_kws={'label': 'Spearman ρ'})
ax.set_title('Base-Agent Signal Correlation (Spearman ρ)\n'
             'Low correlation = genuine diversity', fontweight='bold')
fig.tight_layout()
save_fig(fig, ARTS_DIR / '01_signal_correlation.png')
plt.show()
print(corr_df.corr(method='spearman').round(3).to_string())

In [ ]:
# ── PHASE 3: Walk-forward meta-labeling ───────────────────────────────────────
print('='*60)
print('PHASE 3 — Meta-labeling WFO')
print(f'  Meta-training: {common_start.date()} → {OOS_START.date()}')
print(f'  Meta-OOS     : {OOS_START.date()} → ...')
print('='*60)

meta_agent = MetaSupervisoryAgent(
    sl_mult=1.5,
    tp_mult=2.5,
    max_hold=48,
    train_window=500,
    step_months=3,
    threshold=0.55,
)

t0 = time.time()
sizing, trade_log = meta_agent.run_wfo(
    signals_df,
    oos_start=OOS_START,
    verbose=True,
)
print(f'\nMeta WFO done in {time.time()-t0:.1f}s')

# Sizing summary
meta_oos = sizing[sizing.index >= OOS_START]
n_active = (meta_oos.abs() > 0.05).sum()
print(f'\nMeta sizing summary (OOS {OOS_START.date()} → {meta_oos.index[-1].date()}):')
print(f'  Total bars     : {len(meta_oos):,}')
print(f'  Active (|pos|>0.05): {n_active:,}  ({n_active/len(meta_oos)*100:.1f}%)')
print(f'  Approved trades: {len(trade_log)}')
if len(trade_log):
    print(f'  Long approvals : {(trade_log["side"]==1).sum()}')
    print(f'  Short approvals: {(trade_log["side"]==-1).sum()}')
    print(f'  Mean meta_prob : {trade_log["meta_prob"].mean():.4f}')

In [ ]:
# ── Feature importance of meta-model ─────────────────────────────────────────
fi = meta_agent.get_feature_importance()
if fi is not None:
    print('Meta-model feature importance (last fold):')
    print(fi.head(15).to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 6))
    top_fi = fi.head(15)
    y = range(len(top_fi))
    ax.barh(list(y), top_fi['importance'], color=BLUE, alpha=0.8, edgecolor='white')
    ax.set_yticks(list(y))
    ax.set_yticklabels(top_fi['feature'], fontsize=9)
    ax.set_xlabel('Importance (LGBM gain)')
    ax.set_title('Meta-Model Feature Importance\n(LightGBM — which signal context matters most)',
                 fontweight='bold')
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, ARTS_DIR / '02_meta_feature_importance.png')
    plt.show()

In [ ]:
# ── PHASE 4: Meta-agent backtest ──────────────────────────────────────────────
print('='*60)
print('PHASE 4 — Meta-agent OOS backtest')
print('='*60)

# Run continuous-sizing backtest
meta_oos_sizing = sizing[sizing.index >= OOS_START]
ohlcv_oos = merged[merged.index >= OOS_START][['close','high','low','atr_14_pct']].copy()

eq_meta, tdf_meta_positions = run_sized_backtest(
    meta_oos_sizing, ohlcv_oos, taker_fee=0.0005, funding_h=0.0000077
)

def _sharpe(eq):
    r = np.diff(np.log(np.maximum(eq, 1e-12)))
    return float(r.mean()/(r.std(ddof=1)+1e-12)*np.sqrt(24*365))

def _maxdd(eq):
    pk = np.maximum.accumulate(eq)
    return float(((eq-pk)/(pk+1e-12)).min())

print(f'Meta-Agent OOS results:')
print(f'  Return    : {eq_meta[-1]-1:+.2%}')
print(f'  Sharpe    : {_sharpe(eq_meta):+.3f}')
print(f'  MaxDD     : {_maxdd(eq_meta):.2%}')
print(f'  Pos changes: {len(tdf_meta_positions):,}')

In [ ]:
# ── PHASE 5: Load standalone baselines for comparison ────────────────────────
print('='*60)
print('PHASE 5 — Baseline comparison')
print('='*60)

def _load_results(path):
    try:
        with open(path) as f: return json.load(f)
    except Exception: return None

lgbm_res = _load_results(LGBM_ARTS / 'results.json')
tcn_res  = _load_results(REPO_DIR / 'artifacts' / '06_tcn_omni_0fee_v0' / 'results.json')
drl_res  = _load_results(REPO_DIR / 'artifacts' / '07_drl_omni_0fee_v3' / 'results.json')
gp_res   = _load_results(REPO_DIR / 'artifacts' / '09_gp_omni_0fee_v2' / 'results.json')

print('Standalone results (full OOS w/ fees):')
print(f'{"Agent":<20}  {"Return":>8}  {"Sharpe":>7}  {"MaxDD":>7}  {"Trades":>7}')
print('─'*58)

for name, res in [("LGBM v12", lgbm_res), ("TCN v0", tcn_res), ("DRL v3", drl_res), ("GP v2", gp_res)]:
    if res is None:
        print(f'{name:<20}  (results.json not found — run standalone notebook first)')
    else:
        bt = res.get('backtest_wfees', {})
        print(f'{name:<20}  {bt.get("total_ret",0):>+7.1%}  {bt.get("sharpe",0):>7.3f}  '
              f'{bt.get("maxdd",0):>7.2%}  {bt.get("n_trades",0):>7}')

print(f'{"Meta-Agent v0":<20}  {eq_meta[-1]-1:>+7.1%}  {_sharpe(eq_meta):>7.3f}  '
      f'{_maxdd(eq_meta):>7.2%}  {len(tdf_meta_positions):>7}')

In [ ]:
# ── Overlay equity curves ─────────────────────────────────────────────────────
oos_index = ohlcv_oos.index

# ATH display window
_oos_end_px = ohlcv_oos['close'].iloc[-1]
_ath_mask   = merged['close'] >= _oos_end_px
ATH_START   = merged[_ath_mask].index[0] if _ath_mask.any() else oos_index[0]
_ath_offset = oos_index.searchsorted(ATH_START)
oos_ath     = oos_index[_ath_offset:]

def _rebase(arr): s = arr[_ath_offset:]; return s / s[0]

bh_pct = (ohlcv_oos['close'].iloc[_ath_offset:].values /
           ohlcv_oos['close'].iloc[_ath_offset] - 1) * 100

try:
    _spy = pd.read_parquet(EXT_DIR / 'sp500_daily.parquet')
    _spy.index = pd.to_datetime(_spy.index).tz_localize(None)
    _spy_h = _spy['close'].reindex(oos_ath, method='ffill').ffill().bfill()
    sp500_pct = (_spy_h / _spy_h.iloc[0] - 1) * 100
except Exception:
    sp500_pct = None

fig, ax = plt.subplots(figsize=(14, 6))

# Baselines (if available)
agent_colors = {'LGBM v12': ACCENT, 'TCN v0': TEAL, 'DRL v0': PURPLE}
for name, res, color in [
    ('LGBM v12', lgbm_res, ACCENT),
    ('TCN v0',   tcn_res,  TEAL),
    ('DRL v0',   drl_res,  PURPLE),
]:
    if res is not None:
        bt = res.get('backtest_wfees', {})
        total_ret = bt.get('ath_total_ret', bt.get('total_ret', 0))
        label = f"{name}  Sharpe={bt.get('sharpe',0):.3f}  ret={total_ret:+.1%}"
        ax.plot(oos_ath, [0]*(len(oos_ath)-1) + [0], color=color, lw=0.8, ls='--', alpha=0.5,
                label=label)   # placeholder — real equity requires re-running backtest

meta_eq_ath = _rebase(eq_meta)
ax.plot(oos_ath, (meta_eq_ath-1)*100, color=BLUE, lw=2.0, label=
        f'Meta-Agent  Sharpe={_sharpe(meta_eq_ath):.3f}  ret={meta_eq_ath[-1]-1:+.1%}')
ax.plot(oos_ath, bh_pct, color=GREY, lw=1.0, ls='--', alpha=0.6, label='BTC B&H')
if sp500_pct is not None:
    ax.plot(oos_ath, sp500_pct.values, color='#43A047', lw=0.9, ls=':', alpha=0.7, label='S&P 500')
ax.axhline(0, color=GREY, lw=0.5, ls=':')
ax.set_ylabel('Return (%)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0f}%'))
ax.legend(fontsize=8, loc='upper left')
ax.set_title(f'Multi-Agent Ensemble vs Baselines  |  ATH window {ATH_START.date()}',
             fontweight='bold')
ax.grid(axis='y', alpha=0.25)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.tight_layout()
save_fig(fig, ARTS_DIR / '03_ensemble_comparison.png')
plt.show()

In [ ]:
# ── Monthly returns of meta-agent ─────────────────────────────────────────────
meta_eq_series = pd.Series(eq_meta, index=oos_index)
monthly_eq  = meta_eq_series.resample('ME').last()
monthly_ret = monthly_eq.pct_change().fillna(0) * 100
sma3 = monthly_ret.rolling(3, min_periods=1).mean()

print('Monthly stats (Meta-Agent, OOS w/ fees):')
print(f'  Positive months : {(monthly_ret>0).sum()} / {len(monthly_ret)}')
print(f'  Avg monthly ret : {monthly_ret.mean():+.2f}%')
print(f'  Median monthly  : {monthly_ret.median():+.2f}%')

fig, ax = plt.subplots(figsize=(14, 5))
colors = [GREEN if r >= 0 else RED for r in monthly_ret.values]
ax.bar(monthly_ret.index, monthly_ret.values, color=colors, alpha=0.72, width=22)
ax.plot(monthly_ret.index, sma3.values, color=BLUE, lw=2.0, label='3-month SMA', zorder=5)
ax.axhline(0, color=GREY, lw=0.8, ls=':')
ax.set_ylabel('Return (%)')
ax.set_title(f'Monthly Returns — Meta-Agent v0 (w/ fees)\n'
             f'Positive: {(monthly_ret>0).sum()}/{len(monthly_ret)} months  '
             f'| Avg: {monthly_ret.mean():+.2f}%', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.25)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.tight_layout()
save_fig(fig, ARTS_DIR / '04_monthly_returns.png')
plt.show()

# Heatmap
cal_df = monthly_ret.to_frame('ret').copy()
cal_df['year'] = cal_df.index.year; cal_df['month'] = cal_df.index.month
pivot = cal_df.pivot(index='year', columns='month', values='ret')
pivot.columns = [calendar.month_abbr[m] for m in pivot.columns]
fig2, ax2 = plt.subplots(figsize=(14, max(2, len(pivot)*0.7)))
sns.heatmap(pivot, ax=ax2, cmap='RdYlGn', center=0, annot=True, fmt='.1f',
            linewidths=0.5, linecolor='#e0e0e0', cbar_kws={'label': 'Monthly Return (%)', 'shrink': 0.6},
            annot_kws={'size': 9})
ax2.set_title('Monthly Return Calendar — Meta-Agent v0', fontweight='bold')
ax2.set_xlabel(''); ax2.set_ylabel('')
fig2.tight_layout()
save_fig(fig2, ARTS_DIR / '05_monthly_heatmap.png')
plt.show()

In [ ]:
# ── Save results.json ─────────────────────────────────────────────────────────
results = {
    'notebook': '08_meta_learning_v1',
    'version': 'v1',
    'created': pd.Timestamp.now().isoformat(),
    'approach': 'Meta-Labeling (Lopez de Prado) — multi-agent variant',
    'base_agents': ['lgbm_v12', 'tcn_v0', 'drl_ppo_v3', 'gp_v2'],
    'meta_signal_start': str(META_SIGNAL_START.date()),
    'oos_start': str(OOS_START.date()),
    'oos_end': str(oos_index[-1].date()),
    'meta_config': {
        'sl_mult': meta_agent.sl_mult,
        'tp_mult': meta_agent.tp_mult,
        'max_hold': meta_agent.max_hold,
        'threshold': meta_agent.threshold,
        'step_months': meta_agent.step_months,
        'train_window': meta_agent.train_window,
    },
    'signal_diversity': {
        col: round(float(signals_df[col].corr(signals_df['lgbm_p_up'], method='spearman')), 4)
        for col in ['tcn_p_up', 'tcn_p_down', 'drl_action', 'gp_action']
        if col in signals_df.columns
    },
    'meta_agent': {
        'total_ret': round(float(eq_meta[-1]-1), 4),
        'sharpe': round(_sharpe(eq_meta), 4),
        'maxdd': round(_maxdd(eq_meta), 4),
        'n_position_changes': len(tdf_meta_positions),
        'approved_trades': len(trade_log),
        'mean_position_size': round(float(meta_oos_sizing.abs().mean()), 4),
    },
    'monthly_returns': {
        'mean_pct': round(float(monthly_ret.mean()), 3),
        'median_pct': round(float(monthly_ret.median()), 3),
        'std_pct': round(float(monthly_ret.std()), 3),
        'positive_months': int((monthly_ret > 0).sum()),
        'total_months': int(len(monthly_ret)),
    }
}

out_path = ARTS_DIR / 'results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved → {out_path}')
print(json.dumps(results, indent=2))